# Topic 01 · Titanic mini-EDA with Pandas

A self-contained practice notebook: filtering, sorting, `groupby`, missing values and analytical conclusions.

In [ ]:
import pandas as pd
import numpy as np

## Dataset
For reproducibility this notebook includes a compact representative Titanic sample directly in the notebook. The goal is practicing Pandas mechanics, not reproducing the full competition dataset.

In [ ]:
data = [
(1,0,3,'male',22.0,7.25,'S'),
(2,1,1,'female',38.0,71.2833,'C'),
(3,1,3,'female',26.0,7.925,'S'),
(4,1,1,'female',35.0,53.1,'S'),
(5,0,3,'male',35.0,8.05,'S'),
(6,0,3,'male',np.nan,8.4583,'Q'),
(7,0,1,'male',54.0,51.8625,'S'),
(8,0,3,'male',2.0,21.075,'S'),
(9,1,3,'female',27.0,11.1333,'S'),
(10,1,2,'female',14.0,30.0708,'C'),
(11,1,3,'female',4.0,16.7,'S'),
(12,1,1,'female',58.0,26.55,'S'),
(13,0,3,'male',20.0,8.05,'S'),
(14,0,3,'male',39.0,31.275,'S'),
(15,0,3,'female',14.0,7.8542,'S'),
(16,1,2,'female',55.0,16.0,'S'),
(17,0,3,'male',2.0,29.125,'Q'),
(18,1,2,'male',np.nan,13.0,'S'),
(19,0,3,'female',31.0,18.0,'S'),
(20,1,3,'female',np.nan,7.225,'C'),
]
df = pd.DataFrame(data, columns=['passenger_id','survived','pclass','sex','age','fare','embarked'])
df.head()

## 1. Inspect the frame

In [ ]:
print(df.shape)
print(df.dtypes)
print('\nMissing values:\n', df.isna().sum())

(20, 7)
passenger_id      int64
survived          int64
pclass            int64
sex              object
age             float64
fare            float64
embarked         object
dtype: object

Missing values:
 passenger_id    0
survived        0
pclass          0
sex             0
age             3
fare            0
embarked        0
dtype: int64


## 2. Filter and sort
Female first-class passengers, ordered by fare.

In [ ]:
subset = df.loc[(df['sex']=='female') & (df['pclass']==1), ['age','fare','survived']].sort_values('fare', ascending=False)
print(subset.to_string(index=False))

 age    fare  survived
38.0 71.2833         1
35.0 53.1000         1
58.0 26.5500         1


## 3. Survival rate by sex

In [ ]:
print(df.groupby('sex')['survived'].mean().round(3))

sex
female    0.818
male      0.222
Name: survived, dtype: float64


## 4. Survival rate by passenger class

In [ ]:
print(df.groupby('pclass')['survived'].agg(['count','mean']).round(3))

        count   mean
pclass              
1           4  0.750
2           3  1.000
3          13  0.385


## 5. Missing ages
A common first-pass strategy is median imputation. In a real project, the imputation choice should be justified and ideally fitted only on training data.

In [ ]:
median_age = df['age'].median()
df['age_filled'] = df['age'].fillna(median_age)
print('median age:', median_age)
print('missing after fill:', df['age_filled'].isna().sum())

median age: 31.0
missing after fill: 0


## 6. Derived feature: child

In [ ]:
df['is_child'] = df['age_filled'] < 16
print(df.groupby('is_child')['survived'].agg(['count','mean']).round(3))

          count   mean
is_child              
False        15  0.467
True          5  0.800


## 7. Pivot-style comparison

In [ ]:
print(df.groupby(['sex','pclass'])['survived'].mean().unstack().round(3))

pclass      1      2      3
sex                          
female  1.000  1.000  0.667
male    0.000  1.000  0.125


## Conclusions
This tiny sample is deliberately too small for historical inference, but it is perfect for Pandas practice.

- `groupby(...).mean()` turns a binary target into a rate.
- Compound boolean masks answer precise subgroup questions.
- Missing values must be inspected before analysis.
- Derived features can simplify repeated analytical conditions.
- Multi-index grouping plus `unstack()` gives a compact cross-tab-like view.

The important habit is: **question → subset/group → metric → interpretation**.